# AirWise AI — Premium Dashboard (Veritas AI Hackathon 2026)

**One-click Colab version.** No project-file uploads required. This notebook installs dependencies, writes the complete dashboard source, and launches the interactive Gradio app.

**What it does:** Downloads historical Delhi PM2.5, PM10 and weather from Open-Meteo, trains a 6-hour-ahead Random Forest, compares it against a persistence baseline using a chronological held-out set, and simulates a purifier scheduling policy. It does **not** control a real purifier or measure indoor air.

> **Run the cells in order.** The final cell shows a temporary Gradio link. Initial historical data fetch requires internet access. The displayed latest observation is **historical**, not live.


In [ ]:
!pip -q install "pandas>=2.1,<3" "numpy>=1.26,<3" "scikit-learn>=1.4,<2" "gradio>=5,<7" "plotly>=5.20,<7" "requests>=2.31,<3"

## Dashboard implementation
The following cell contains the **entire app**. Change the design, model, and copy here; no external `.py` uploads are necessary.

In [ ]:
%%writefile airwise_app.py
"""AirWise AI — self-contained historical PM2.5 forecasting dashboard.
Sources: Open-Meteo air-quality and weather historical endpoints.
A prototype, NOT a real-time or indoor safety controller.
"""
import datetime as dt
import functools
import html
import os
import warnings
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import gradio as gr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings('ignore', category=FutureWarning)

LAT, LON = 28.6139, 77.2090
TZ = 'Asia/Kolkata'
HORIZON = 6

CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap');
:root{--bg:#080f1c;--panel:#101d30;--stroke:#233249;--mint:#5ee9bc;--muted:#8fa3bd}
.gradio-container{font-family:'DM Sans',sans-serif!important;max-width:1440px!important;margin:auto!important;background:#080f1c!important;color:#f4f8ff!important}
body,.gradio-container,.main{background:#080f1c!important}
#shell{padding:0 14px 28px}
#header{margin-top:10px;margin-bottom:20px}
#header .prose{max-width:none}
.logo-row{display:flex;align-items:center;justify-content:space-between;gap:12px;flex-wrap:wrap;border-bottom:1px solid #26354a;padding:18px 4px 20px}
.brand{display:flex;align-items:center;gap:12px;font:800 22px Manrope,sans-serif;letter-spacing:-1px;color:#fff}
.brand-icon{display:grid;place-items:center;width:38px;height:38px;border-radius:12px;background:#5ee9bc;color:#0b1824;font-size:22px}
.brand span{color:#5ee9bc}.chip{border:1px solid #2a4450;background:#102b34;border-radius:100px;padding:8px 12px;color:#8ceacb;font-size:12px;font-weight:700;letter-spacing:.03em}
.hero{padding:12px 3px 22px}.eyebrow{font-size:12px;text-transform:uppercase;letter-spacing:2.3px;font-weight:800;color:#65ddbb}.hero h1{font:800 clamp(28px,4vw,43px) Manrope,sans-serif;letter-spacing:-2px;line-height:1.13;color:#f7fbff;margin:9px 0 10px}.hero p{color:#94a7bf;font-size:14px;max-width:740px;line-height:1.65}
#config,#forecastPanel,#methodsPanel,#chartPanel,#comparisonPanel{background:#101d30!important;border:1px solid #24344c!important;border-radius:20px!important;padding:20px!important;box-shadow:0 12px 38px #00000016}
.section-heading{font:800 16px Manrope,sans-serif;color:#f6faff;margin:0 0 5px}.section-sub{font-size:12px;color:#879bb5;line-height:1.5;margin-bottom:14px}.small-label{font-size:12px;color:#91a4bb}
#config label span,#config label,#config .wrap{color:#d6e4f5!important}#config input{color:#eff9ff!important}
#action{background:linear-gradient(100deg,#58e6b8,#80efd0)!important;color:#0a1b25!important;font-weight:800!important;border:0!important;border-radius:12px!important;min-height:48px}
.metrics{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:13px;margin-bottom:14px}.metric{border:1px solid #263851;border-radius:16px;background:#121f33;padding:18px}.metric .label{font-size:12px;color:#92a7c0}.metric .value{font:800 28px Manrope,sans-serif;color:#f8fbff;letter-spacing:-1.4px;margin:9px 0 4px}.metric .unit{font:500 12px 'DM Sans';color:#90a6bf;letter-spacing:0}.metric .note{font-size:12px;color:#8094ac}.metric.accent{background:linear-gradient(135deg,#103838,#153149);border-color:#276b66}.metric.accent .value{color:#70f0c9}
.verdict{padding:23px;border:1px solid #3c6353;border-radius:18px;background:linear-gradient(120deg,#10372f,#102d37 70%,#102236);margin-bottom:16px;display:flex;align-items:center;justify-content:space-between;gap:20px;flex-wrap:wrap}.verdict.off{background:linear-gradient(120deg,#183b36,#142d39);border-color:#366354}.verdict .status{font:800 23px Manrope,sans-serif;color:#c2ffe8}.verdict .desc{color:#aed0c8;font-size:13px;max-width:600px;line-height:1.65;margin-top:6px}.verdict .symbol{font-size:34px;color:#71efbd}
.report-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:13px}.mini{padding:15px;background:#15243a;border:1px solid #2b3b52;border-radius:14px}.mini b{display:block;color:#fff;font-size:20px;margin:7px 0}.mini span{color:#9fb0c6;font-size:12px}.method-list{color:#aabbd0;font-size:13px;line-height:1.9}.method-list strong{color:#e4f3ff}.smallprint{font-size:11px;color:#8599b1;line-height:1.7;margin-top:12px}.error{color:#ffb7a9;border:1px solid #86504b;background:#301e25;padding:18px;border-radius:14px}
#forecastChart{background:transparent!important}#forecastChart .plot-container{border-radius:16px;overflow:hidden}
.gradio-container button:not(#action){border-radius:12px!important}
@media(max-width:850px){.metrics{grid-template-columns:repeat(2,1fr)}.report-grid{grid-template-columns:1fr}.metric .value{font-size:23px}}
@media(max-width:460px){.metrics{grid-template-columns:1fr 1fr;gap:8px}.metric{padding:12px}.metric .value{font-size:21px}}
"""

def get_json(url, params):
    response = requests.get(url, params=params, timeout=35, headers={'User-Agent':'AirWiseAI-Hackathon/1.0'})
    response.raise_for_status()
    payload=response.json()
    if 'hourly' not in payload:
        raise RuntimeError(payload.get('reason', 'No hourly data returned'))
    return payload['hourly']

def load_data():
    # Explicit historical dates make the demonstration reproducible and do not imply live data.
    end = dt.date.today() - dt.timedelta(days=7)
    start = end - dt.timedelta(days=100)
    dates={'latitude':LAT,'longitude':LON,'start_date':str(start),'end_date':str(end),'timezone':TZ}
    air = get_json('https://air-quality-api.open-meteo.com/v1/air-quality', {
        **dates,'hourly':'pm2_5,pm10'})
    weather = get_json('https://archive-api.open-meteo.com/v1/archive', {
        **dates,'hourly':'temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation'})
    a=pd.DataFrame(air).rename(columns={'pm2_5':'pm25'})
    w=pd.DataFrame(weather)
    a['time']=pd.to_datetime(a['time']); w['time']=pd.to_datetime(w['time'])
    d=a.merge(w,on='time',how='inner').sort_values('time').reset_index(drop=True)
    for col in ['pm25','pm10','temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation']:
        d[col]=pd.to_numeric(d[col],errors='coerce')
    d=d.dropna(subset=['pm25']).copy()
    d=d.set_index('time').asfreq('h')
    # Interpolate short gaps only; do not make large gaps look like measured data.
    d[['pm25','pm10','temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation']]=d[['pm25','pm10','temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation']].interpolate(limit=2)
    d=d.reset_index()
    if len(d)<250:raise RuntimeError('Insufficient hourly historical data for evaluation')
    return d

FEATURES=['pm25','pm10','temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation','hour_sin','hour_cos','day_sin','day_cos','pm_lag_1','pm_lag_3','pm_lag_6','pm_roll_6']

def make_features(df):
    d=df.copy()
    h=d.time.dt.hour
    d['hour_sin']=np.sin(2*np.pi*h/24);d['hour_cos']=np.cos(2*np.pi*h/24)
    day=d.time.dt.dayofyear
    d['day_sin']=np.sin(2*np.pi*day/365.25);d['day_cos']=np.cos(2*np.pi*day/365.25)
    for n in [1,3,6]:d[f'pm_lag_{n}']=d.pm25.shift(n)
    d['pm_roll_6']=d.pm25.shift(1).rolling(6).mean()
    d['target']=d.pm25.shift(-HORIZON)
    return d

@functools.lru_cache(maxsize=1)
def trained():
    raw=load_data()
    featured=make_features(raw)
    all_rows=featured.dropna(subset=FEATURES+['target']).copy()
    # Chronological 70% train, 10% validation, 20% untouched test.
    n=len(all_rows);tr=int(n*.70);va=int(n*.80)
    train=all_rows.iloc[:tr-HORIZON]; validation=all_rows.iloc[tr:va-HORIZON]; test=all_rows.iloc[va:]
    # Hyperparameters are fixed; validation is retained for inspection and not used in tuning.
    model=RandomForestRegressor(n_estimators=180,max_depth=14,min_samples_leaf=3,random_state=42,n_jobs=-1)
    model.fit(train[FEATURES],train['target'])
    test=test.copy();test['prediction']=np.maximum(0,model.predict(test[FEATURES]))
    test['baseline']=test['pm25']
    model_mae=mean_absolute_error(test.target,test.prediction)
    baseline_mae=mean_absolute_error(test.target,test.baseline)
    latest=featured.dropna(subset=FEATURES).iloc[-1]
    prediction=max(0,float(model.predict(pd.DataFrame([latest[FEATURES]])[FEATURES])[0]))
    return {'raw':raw,'test':test,'latest':latest,'prediction':prediction,'model_mae':model_mae,'baseline_mae':baseline_mae,'validation_count':len(validation),'train_count':len(train),'test_count':len(test)}

def esc(x):return html.escape(str(x))

def metric(label,value,note='',accent=False):
    return f'<div class="metric {"accent" if accent else ""}"><div class="label">{label}</div><div class="value">{value}</div><div class="note">{note}</div></div>'

def chart(result,trigger):
    d=result['raw'].dropna(subset=['pm25']).tail(60)
    last=result['latest'];forecast_time=last.time+pd.Timedelta(hours=HORIZON)
    fig=go.Figure()
    fig.add_trace(go.Scatter(x=d.time,y=d.pm25,mode='lines',name='Observed PM2.5',line=dict(color='#62e6bd',width=3),fill='tozeroy',fillcolor='rgba(94,233,188,.06)',hovertemplate='%{x|%d %b %H:%M}<br>%{y:.1f} µg/m³<extra></extra>'))
    fig.add_trace(go.Scatter(x=[last.time,forecast_time],y=[last.pm25,result['prediction']],mode='lines+markers',name='AI six-hour forecast',line=dict(color='#f6b46e',width=3,dash='dash'),marker=dict(size=[8,13]),hovertemplate='%{x|%d %b %H:%M}<br>%{y:.1f} µg/m³<extra></extra>'))
    fig.add_hline(y=float(trigger),line_dash='dot',line_color='#8d9fba',annotation_text='Selected trigger',annotation_font_color='#b9c5d7')
    fig.update_layout(height=355,margin=dict(l=22,r=22,t=18,b=20),paper_bgcolor='#101d30',plot_bgcolor='#101d30',font=dict(color='#91a7be',family='DM Sans'),xaxis=dict(showgrid=False,zeroline=False,title=None),yaxis=dict(gridcolor='#23334a',zeroline=False,title='PM2.5 · µg/m³'),legend=dict(orientation='h',y=1.18,x=0),hovermode='x unified')
    return fig

def render(power,tariff,trigger):
    try:
        power=float(power);tariff=float(tariff);trigger=float(trigger)
        if not(5<=power<=2000 and 0<=tariff<=100 and 0<=trigger<=1000):raise ValueError('Please provide valid power, tariff and trigger values.')
        r=trained();test=r['test'];latest=r['latest'];pred=r['prediction'];on=pred>=trigger
        actual_bad=test.target>=trigger
        ai_on=test.prediction>=trigger
        rule_on=test.pm25>=trigger
        hours=len(test);month_hours=720
        ai_fraction=float(ai_on.mean());base_fraction=float(rule_on.mean())
        always_kwh=power*month_hours/1000
        scheduled_kwh=always_kwh*ai_fraction
        saved_kwh=always_kwh-scheduled_kwh
        saved_rupees=saved_kwh*tariff
        missed=int((actual_bad & ~ai_on).sum());covered=int((actual_bad & ai_on).sum()); total_bad=int(actual_bad.sum())
        baseline_missed=int((actual_bad & ~rule_on).sum())
        ts=pd.Timestamp(latest.time)
        target_time=ts+pd.Timedelta(hours=HORIZON)
        cards='<div class="metrics">'+''.join([
            metric('Latest stored PM2.5',f'{latest.pm25:.1f}<span class="unit"> µg/m³</span>',f'Observed · {ts:%d %b %Y, %H:%M}'),
            metric('AI forecast · +6h',f'{pred:.1f}<span class="unit"> µg/m³</span>',f'Target · {target_time:%d %b, %H:%M}',True),
            metric('Monthly saving potential',f'₹{saved_rupees:,.0f}',f'{saved_kwh:.1f} kWh · simulated'),
            metric('Held-out model MAE',f'{r["model_mae"]:.1f}<span class="unit"> µg/m³</span>',f'Baseline: {r["baseline_mae"]:.1f} µg/m³')
        ])+'</div>'
        title='KEEP PURIFIER ON' if on else 'BELOW SELECTED TRIGGER'
        desc=(f'Forecast PM2.5 ({pred:.1f} µg/m³) is above the selected {trigger:.0f} µg/m³ threshold.' if on else f'Forecast PM2.5 ({pred:.1f} µg/m³) is below the selected {trigger:.0f} µg/m³ threshold. Do not automatically turn a real purifier off based on outdoor air alone.')
        verdict=f'<div class="verdict {"" if on else "off"}"><div><div class="eyebrow">Six-hour recommendation · simulation only</div><div class="status">{title}</div><div class="desc">{esc(desc)}<br>Indoor air quality is not measured; this is not a safety instruction.</div></div><div class="symbol">{"↗" if on else "◎"}</div></div>'
        energy='<div class="report-grid">'+''.join([
            f'<div class="mini"><span>Always running · 30 days</span><b>{always_kwh:.1f} kWh</b><span>₹{always_kwh*tariff:,.0f} estimated cost</span></div>',
            f'<div class="mini"><span>AI scheduled · 30 days</span><b>{scheduled_kwh:.1f} kWh</b><span>₹{scheduled_kwh*tariff:,.0f} estimated cost</span></div>',
            f'<div class="mini"><span>Estimated runtime reduction</span><b>{(1-ai_fraction)*100:.1f}%</b><span>Equivalent monthly saving: ₹{saved_rupees:,.0f}</span></div>'
        ])+'</div>'
        performance='<div class="report-grid">'+''.join([
            f'<div class="mini"><span>Model error (MAE)</span><b>{r["model_mae"]:.1f} µg/m³</b><span>Random Forest · held-out test</span></div>',
            f'<div class="mini"><span>Persistence baseline error</span><b>{r["baseline_mae"]:.1f} µg/m³</b><span>Assumes PM2.5 stays unchanged</span></div>',
            f'<div class="mini"><span>High-pollution test hours covered</span><b>{covered} / {total_bad}</b><span>{missed} missed · chosen trigger</span></div>',
        ])+'</div>'+f'<p class="smallprint">Test size: {r["test_count"]} hourly predictions. Current-reading rule missed {baseline_missed} high-pollution target hours; AI forecast rule missed {missed}. Estimated energy saved by current-reading rule: {(1-base_fraction)*100:.1f}% versus {100*(1-ai_fraction):.1f}% by the AI rule. These are historical outdoor-air simulations, not observed indoor exposure or measured electricity savings.</p>'
        methodology=f'<div class="method-list"><strong>Model:</strong> Random Forest Regressor · 180 trees · 6-hour forecast horizon<br><strong>Inputs:</strong> PM2.5, PM10, temperature, humidity, wind, precipitation, time cycles, lagged PM2.5<br><strong>Source:</strong> Open-Meteo archived Delhi air-quality and weather data<br><strong>Evaluation:</strong> chronological 70% train / 10% validation / 20% untouched test; train {r["train_count"]:,} rows, validation {r["validation_count"]:,} rows, test {r["test_count"]:,} rows<br><strong>Policy:</strong> forecast PM2.5 ≥ chosen threshold means projected purifier ON; comparison uses actual six-hour-ahead outdoor PM2.5<br><strong>Data timestamp:</strong> latest archived observation {ts:%d %B %Y, %H:%M} (Asia/Kolkata)</div><p class="smallprint">Research prototype. This interface retrieves historical data when the Colab session starts, not a live PM2.5 measurement. Changing the threshold is exploratory and does not constitute independently validated control-policy optimization. No indoor PM2.5 measurements, device control, standby energy, or variable fan speeds are included. Avoid switching off devices solely on this prediction.</p>'
        return cards,verdict,chart(r,trigger),energy,performance,methodology
    except Exception as exc:
        msg=f'<div class="error"><strong>Unable to calculate the forecast</strong><br>{esc(exc)}<br>Check Colab internet access and try again. This demo needs Open-Meteo historical API access.</div>'
        return msg,'',go.Figure(),'','',''

with gr.Blocks(title='AirWise AI · Delhi Pollution Forecast',css=CSS,theme=gr.themes.Base(primary_hue='emerald',neutral_hue='slate')) as demo:
    with gr.Column(elem_id='shell'):
        gr.HTML('''<div id="header"><div class="logo-row"><div class="brand"><div class="brand-icon">◈</div> AirWise <span>AI</span></div><div class="chip">● AI + CLIMATE · HISTORICAL DEMO</div></div></div>''')
        gr.HTML('''<div class="hero"><div class="eyebrow">DELHI · AIR QUALITY INTELLIGENCE</div><h1>Cleaner air. <span style="color:#69eac0">Smarter energy.</span></h1><p>Explore a machine-learning forecast of outdoor particulate pollution and simulate when an air purifier could operate. Built for the Veritas AI International High School Hackathon 2026.</p></div>''')
        with gr.Row(equal_height=False):
            with gr.Column(scale=1,min_width=280,elem_id='config'):
                gr.HTML('<div class="section-heading">Your purifier setup</div><div class="section-sub">Adjust assumptions to explore forecast-based energy use.</div>')
                power=gr.Slider(20,200,value=60,step=5,label='Purifier power (W)')
                tariff=gr.Number(value=8,precision=2,label='Electricity price (₹/kWh)')
                trigger=gr.Slider(30,150,value=60,step=5,label='PM2.5 trigger (µg/m³)')
                run=gr.Button('✦  Run AI forecast & analysis',variant='primary',elem_id='action')
                gr.HTML('<div class="smallprint">Forecasts are evaluated against stored historical observations, not a live sensor. The threshold is a configurable experiment, not a medical or safety standard.</div>')
            with gr.Column(scale=3,min_width=380,elem_id='forecastPanel'):
                cards=gr.HTML()
                recommendation=gr.HTML()
                gr.HTML('<div class="section-heading">Pollution trend & forecast</div><div class="section-sub">Recent historical readings with one six-hour-ahead prediction</div>')
                figure=gr.Plot(elem_id='forecastChart',show_label=False)
        with gr.Tabs():
            with gr.Tab('⚡  Energy simulator'):
                with gr.Column(elem_id='comparisonPanel'):
                    gr.HTML('<div class="section-heading">What could smarter scheduling save?</div><div class="section-sub">Extrapolated from the forecast-rule operating fraction on the test period.</div>')
                    energy_html=gr.HTML()
            with gr.Tab('◎  Model performance'):
                with gr.Column(elem_id='comparisonPanel'):
                    gr.HTML('<div class="section-heading">Measured against a simple baseline</div><div class="section-sub">A separate, chronologically held-out test period provides a fairer comparison.</div>')
                    performance_html=gr.HTML()
            with gr.Tab('ⓘ  Data & methodology'):
                with gr.Column(elem_id='methodsPanel'):
                    gr.HTML('<div class="section-heading">How this prototype works</div>')
                    method_html=gr.HTML()
        gr.HTML('<p class="smallprint" style="padding:12px 4px">AirWise AI · Educational research prototype · Weather/air quality: Open-Meteo · Powered by scikit-learn, Plotly, Gradio.</p>')
    outputs=[cards,recommendation,figure,energy_html,performance_html,method_html]
    run.click(fn=render,inputs=[power,tariff,trigger],outputs=outputs)
    demo.load(fn=render,inputs=[power,tariff,trigger],outputs=outputs)

if __name__=='__main__':
    demo.launch(share=os.getenv('GRADIO_SHARE','false').lower()=='true',show_error=True)


## Launch the dashboard
The link is temporary and works while this Colab runtime remains running. The first forecast may require a data download and model training.

In [ ]:
import os
os.environ['GRADIO_SHARE'] = 'true'
%run airwise_app.py